# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
os.environ.pop("SSLKEYLOGFILE", None)
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI



In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
ollama_base_url = os.getenv('OLLAMA_BASE_URL')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'llama3.2'
#openai = OpenAI()
ollama = OpenAI(base_url=ollama_base_url, api_key='ollama')

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2
Found 3 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula'},
  {'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'posts/courses', 'url': 'https://edwarddonner.com/posts/'}]}

In [8]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [12]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2


d:\AI Learn\llm_engineering\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'edwarddonner.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Found 4 relevant links


{'links': [{'type': 'Company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'About page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'Posts/Courses', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'LinkedIn profile',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [13]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2


d:\AI Learn\llm_engineering\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Found 13 relevant links


{'links': [{'type': 'brand webpage', 'url': 'https://huggingface.co'},
  {'type': 'company page', 'url': 'https://huggingface.co/brand'},
  {'type': 'about page', 'url': 'https://huggingface.co/about'},
  {'type': 'GitHub repository', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter account', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn company page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Discord server (for Hugging Face developers)',
   'url': 'https://join.discord.gg/huggingface'},
  {'type': 'Blog', 'url': 'https://blog.huggingface.co'},
  {'type': 'Changelog', 'url': 'https://huggingface.co/changelog'},
  {'type': 'GitHub repository (specific models)',
   'url': 'https://github.com/huggingface/models'},
  {'type': 'Documentation for Hugging Face models',
   'url': 'https://huggingface.co/docs/models'},
  {'type': 'Documentation for Transformers library',
   'url': 'https://huggingface.co/docs/transformers'},
  {'type

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [28]:
def fetch_page_and_all_relevant_links(url):
    try: 
        contents = fetch_website_contents(url)
        relevant_links = select_relevant_links(url)
        result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
        for link in relevant_links['links']:
            result += f"\n\n### Link: {link['type']}\n"
            result += fetch_website_contents(link["url"])
        return result
    except Exception as e: 
        return ""

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 6 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
google/gemma-4-12B-it
Updated
6 days ago
•
581k
•
855
nvidia/LocateAnything-3B
Updated
1 day ago
•
124k
•
1.78k
unsloth/gemma-4-12b-it-GGUF
Updated
about 21 hours ago
•
660k
•
541
google/gemma-4-12B
Updated
6 days ago
•
122k
•
494
ideogram-ai/ideogram-4-fp8
Updated
7 days ago
•
5.92k
•
457
Browse

In [15]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 11 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ngoogle/gemma-4-12B-it\nUpdated\n6 days ago\n•\n676k\n•\n866\nnvidia/LocateAnything-3B\nUpdated\n2 days ago\n•\n132k\n•\n1.79k\

In [26]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [29]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 6 relevant links


# HuggingFace: Revolutionizing Natural Language Processing

[Overview]
-----------

HuggingFace is a cutting-edge technology company that specializes in natural language processing (NLP) and artificial intelligence. Founded in 2016, the company has quickly become one of the leading voices in the field.

[Problem Statement]
--------------------

The world's data is increasingly written in text form, yet most AI systems struggle to understand and interpret this information. HuggingFace aims to change this by providing open-source tools and platforms for building, training, and deploying NLP models.

[Solutions]
------------

*   [Transformers Hub](https://huggingface.co/transformers): A comprehensive hub offering pre-trained models for a wide range of NLP tasks.
*   **Model Training & Deployment**: HuggingFace's AutoML feature allows users to easily train, deploy, and manage their own NLP models on a scalable platform.
*   [Dataset Hub](https://huggingface.co/datasets): A collaborative dataset repository providing access to millions of text files for model training.

[Company Culture]
------------------

At HuggingFace, we are passionate about harnessing the power of AI to transform industries and lives. **Our culture**:

*   Prioritizes collaboration: Our team is built on a foundation of camaraderie and collective knowledge-sharing.
*   Foster creativity: Encouraging innovative thinking and outside-the-box solutions that push the boundaries of what's possible with NLP.
*   Thrives in autonomy: Empowering individuals to work independently, focusing on their areas of strength.

[Customer Base]
----------------

HuggingFace serves a diverse range of industries and organizations across the globe. **Notable Customers**, such as:

*   Data.gov (American government dataset platform)
*   MIT CSAIL's NLP Lab
*   Microsoft

[Careers & Jobs]
--------------

Join our highly innovative environment:

*   [Join Our Team](https://careers.huggingface.com/): Explore current and upcoming job openings at HuggingFace.

If you have any questions or would like to proceed, please let me know.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [33]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [35]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 8 relevant links


# Hugging Face
Unlocking Human Intelligence with AI

At Hugging Face, we believe that artificial intelligence should be used to augment human capabilities, not supplant them. Our vision is to make it easy for anyone to use AI, without requiring extensive technical expertise.

## About Us

Our mission is to democratize access to pre-trained models and create a community-driven platform where researchers, developers, and practitioners can share, collaborate, and showcase their work.

We're passionate about building tools that help humans better understand and interact with language, vision, and other forms of human intelligence. From natural language processing (NLP) to computer vision, we're exploring the frontiers of human-AI collaboration.

## Our Mission

We aim to empower anyone – academics, entrepreneurs, artists, and everyday people – to build AI-powered applications that make a positive impact on society.

## Community-Driven Approach

Our approach is built around three core pillars:

*   **Research**: We foster a culture of open research, where experts from academia and industry collaborate on high-quality projects.
*   **Open-Source**: We prioritize open-source innovations, making our tools and models accessible to everyone.
*   **Ecosystem**: We build a thriving ecosystem of developers, researchers, and practitioners who share ideas, resources, and knowledge.

## Our Successes

We've already helped thousands of users build cutting-edge AI applications, including popular projects like:

*   [Transformers](https://huggingface.co/transformers/)
*   [DistilBERT](https://huggingface.co/distilbert)
*   [Open-Datasets](https://huggingface.co/datasets)

## Join Our Mission

Are you passionate about human-AI collaboration? We have opportunities for:

*   **Researchers**: Share your expertise and contribute to open-source projects.
*   **Developers**: Build AI-powered applications that make a difference.
*   **Careers**: Explore job openings in research, development, and support.

[Learn more about our careers](javascript:void(0))

## Join the Movement

Stay up-to-date with the latest Hugging Face news, blog posts, and updates:

[Blog](javascriptVoid\(\))
[Twitter](https://twitter.com/huggingface)

Let's collaborate and unlock human intelligence together.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>